# Notebook 02 · Preprocessing & Data Cleaning
**Input :** `F:\mmd\data\processed\reviews\*.parquet` + `meta\*.parquet`  
**Output:** `F:\mmd\data\cleaned\reviews_clean.parquet`  
           `F:\mmd\data\cleaned\meta_clean.parquet`  
           `F:\mmd\data\cleaned\sessions.parquet` ← user → sorted item sequence  

---
### Pipeline
```
reviews/          meta/
  │  lazy scan      │  lazy scan
  ▼                 ▼
dedup            dedup parent_asin
  │              parse price → float
filter verified_purchase=True
  │
filter rating >= 1 (all keep, implicit feedback)
  │
k-core filtering  (user ≥5, item ≥5 interactions)
  │                        ↑ lặp đến hội tụ
  ▼
build sessions   (group by user_id, sort timestamp)
  │
temporal split   train / val / test
  ▼
lưu parquet
```

In [2]:
import psutil
from tqdm import tqdm

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"{ram_usage()}")

RAM: 5.0/7.9 GB (64%)


## 0 · Imports & config

In [3]:
import os, gc, sys, logging
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import psutil
from tqdm.auto import tqdm

import numpy as np
from pathlib import Path


ROOT_DIR      = Path(r"F:\amazon_data") 
REVIEW_PATH   = ROOT_DIR / "data" / "json" / "review_Home_and_Kitchen.jsonl"
META_PATH     = ROOT_DIR / "data" / "json" / "meta_Home_and_Kitchen.jsonl"

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
TEMP_DIR      = ROOT_DIR / "data" / "processed" / "_temp_chunks"

for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    
CHUNK_SIZE           = 200_000
MIN_TEXT_LENGTH      = 10
MIN_REVIEWS_PER_USER = 5
MIN_REVIEWS_PER_ITEM = 5
RANDOM_SEED          = 42

np.random.seed(RANDOM_SEED)

print("=== CẤU HÌNH & THÔNG TIN FILE ===")
print(f"   Chunk size : {CHUNK_SIZE:,} dòng/lần")

if REVIEW_PATH.exists():
    print(f"   Review file: {REVIEW_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Review file: Không tìm thấy tại {REVIEW_PATH}")

if META_PATH.exists():
    print(f"   Meta file  : {META_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Meta file  : Không tìm thấy tại {META_PATH}")

print(f"   RAM: {ram_usage()}")

=== CẤU HÌNH & THÔNG TIN FILE ===
   Chunk size : 200,000 dòng/lần
   Review file: 29.25 GB
   Meta file  : 10.98 GB
   RAM: RAM: 5.2/7.9 GB (66%)


## 1 · Load reviews (lazy)

In [3]:
gc.collect()

0

In [4]:
def count_lines(filepath):
    count = 0
    with open(filepath, 'rt', encoding='utf-8') as f:
        for _ in tqdm(f, desc=filepath.name[:35], unit=" lines"):
            count += 1
    return count

n_reviews = count_lines(REVIEW_PATH)
n_meta    = count_lines(META_PATH)
n_chunks  = (n_reviews // CHUNK_SIZE) + 1

print(f"   Review: {n_reviews:,} dòng → {n_chunks} chunks")
print(f"   Meta  : {n_meta:,} dòng")

review_Home_and_Kitchen.jsonl: 0 lines [00:00, ? lines/s]

meta_Home_and_Kitchen.jsonl: 0 lines [00:00, ? lines/s]

   Review: 67,409,944 dòng → 338 chunks
   Meta  : 3,735,584 dòng


## 2 · Clean Reviews
### 2.1 Load + dedup + filter verified

### 2.2 K-core filtering
Lặp lọc đến khi hội tụ: user ≥ 5 tương tác **và** item ≥ 5 tương tác.

In [5]:

def clean_review_chunk(df):
    if df.empty:
        return None

    # 1. Lọc cột
    needed = ['rating', 'text', 'title', 'user_id',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    df = df[[c for c in needed if c in df.columns]].copy()

    # 2. Drop các dòng thiếu dữ liệu cốt lõi
    df = df.dropna(subset=['rating', 'text'])
    if df.empty: return None

    # 3. Chuẩn hóa kiểu dữ liệu số
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df = df.dropna(subset=['rating']) # Lọc lại lần nữa nếu coerce tạo ra NaN
    if df.empty: return None
    
    df['rating'] = df['rating'].astype('float32')
    
    if 'helpful_vote' in df.columns:
        df['helpful_vote'] = pd.to_numeric(df['helpful_vote'], errors='coerce').fillna(0).astype('int32')
        
    if 'verified_purchase' in df.columns:
        df['verified_purchase'] = df['verified_purchase'].fillna(False).astype(bool)

    # 4. Timestamp → year, month
    if 'timestamp' in df.columns:
        dt = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = dt.dt.year.astype('Int16')
        df['month'] = dt.dt.month.astype('Int8')

    # 5. Tạo nhãn Sentiment bằng Numpy 
    conditions = [
        df['rating'] >= 4,
        df['rating'] == 3
    ]
    choices = ['positive', 'neutral']
    df['sentiment'] = np.select(conditions, choices, default='negative')
    df['sentiment'] = df['sentiment'].astype('category')

    # 6. Làm sạch Text và lọc độ dài (Loại bỏ review chỉ có dấu cách)
    # Loại bỏ thẻ html <br> hay gặp trong review Amazon
    df['text'] = df['text'].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True).str.strip()
    df['text_length'] = df['text'].str.len().astype('int32')
    
    df = df[df['text_length'] >= MIN_TEXT_LENGTH]
    if df.empty: return None

    # 7. Loại duplicate trong cùng chunk
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')

    return df.reset_index(drop=True)

In [6]:
import gc

for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

chunk_id      = 0
total_raw     = 0
total_clean   = 0
current_chunk = []

print(f"   Chunk size: {CHUNK_SIZE:,} dòng/lần\n")

# 2. Đọc file text thường (không dùng gzip)
with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:

    pbar = tqdm(f, desc="Processing", unit=" lines")

    for line in pbar:
        total_raw += 1
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue

        # Đủ 1 chunk thì đem đi xử lý
        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk       = pd.DataFrame(current_chunk)
            df_chunk_clean = clean_review_chunk(df_chunk)

            if df_chunk_clean is not None:
                chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
                df_chunk_clean.to_parquet(chunk_path, index=False)
                total_clean += len(df_chunk_clean)

            chunk_id      += 1
            current_chunk  = []
            
            # Xóa biến và dọn dẹp RAM
            del df_chunk, df_chunk_clean
            gc.collect()  

            # Cập nhật thông tin lên thanh tiến trình
            pbar.set_postfix({
                'chunks': chunk_id,
                'clean' : f"{total_clean:,}",
                'RAM'   : f"{psutil.virtual_memory().percent:.0f}%"
            })

    # 3. Xử lý phần dư cuối file (nếu tổng dòng không chia hết cho CHUNK_SIZE)
    if current_chunk:
        df_chunk       = pd.DataFrame(current_chunk)
        df_chunk_clean = clean_review_chunk(df_chunk)
        
        if df_chunk_clean is not None:
            chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
            df_chunk_clean.to_parquet(chunk_path, index=False)
            total_clean += len(df_chunk_clean)
        chunk_id += 1

# Dọn rác lần cuối
gc.collect()

print("\n=== KẾT QUẢ XỬ LÝ ===")
print(f"   Tổng đọc  : {total_raw:,} dòng")
print(f"   Sau sạch  : {total_clean:,} dòng ({total_clean/total_raw*100:.1f}% giữ lại)")
print(f"   Số chunks : {chunk_id}")

   Chunk size: 200,000 dòng/lần



Processing: 0 lines [00:00, ? lines/s]


=== KẾT QUẢ XỬ LÝ ===
   Tổng đọc  : 67,409,944 dòng
   Sau sạch  : 0 dòng (0.0% giữ lại)
   Số chunks : 0


In [7]:
import gc
import shutil
import pyarrow.parquet as pq
from tqdm import tqdm 

chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean.parquet"

print(f"Đang gộp {len(chunk_files)} chunks lại thành một file duy nhất...\n")

writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    # Đọc chunk dưới dạng Arrow Table
    table = pq.read_table(chunk_file)
    
    # Khởi tạo writer dựa trên schema của chunk đầu tiên
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
        
    writer.write_table(table)
    
    # Dọn dẹp RAM
    del table
    gc.collect()

if writer:
    writer.close()

# Xóa thư mục chunks tạm
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)

# Kiểm tra kết quả
if output_path.exists():
    sz = output_path.stat().st_size / (1024**3)
    print(f"\n Đã lưu file cuối cùng tại: {output_path}")
    print(f" Dung lượng review_clean.parquet: {sz:.2f} GB")
else:
    print("\n Lỗi: Không tìm thấy file output sau khi gộp.")

Đang gộp 0 chunks lại thành một file duy nhất...



Merging chunks: 0it [00:00, ?it/s]


 Đã lưu file cuối cùng tại: F:\amazon_data\data\processed\review_clean.parquet
 Dung lượng review_clean.parquet: 9.75 GB


## 3 · Clean Meta
Parse price, dedup `parent_asin`, filter chỉ giữ item có trong reviews.

In [8]:
def _clean_meta_df(df):
    """Làm sạch 1 chunk meta"""
    if df is None or df.empty:
        return None

    needed = ['parent_asin','title','price','description',
              'categories','average_rating','rating_number',
              'store','main_category']
    df = df[[c for c in needed if c in df.columns]].copy()

    # Xóa dòng thiếu ID hoặc Tên sản phẩm, và xóa trùng lặp
    df = df.dropna(subset=['parent_asin','title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')

    # Xử lý giá tiền (chỉ lấy số và dấu thập phân)
    if 'price' in df.columns:
        df['price'] = pd.to_numeric(
            df['price'].astype(str).str.replace(r'[^\d.]','',regex=True),
            errors='coerce'
        ).astype('float32')

    # Xử lý rating để đồng bộ kiểu dữ liệu
    if 'average_rating' in df.columns:
        df['average_rating'] = pd.to_numeric(df['average_rating'], errors='coerce').astype('float32')
    if 'rating_number' in df.columns:
        df['rating_number'] = pd.to_numeric(df['rating_number'], errors='coerce').astype('Int32')

    # Xử lý description (nối list thành chuỗi)
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else '')
        )

    # Rút trích danh mục chính từ mảng categories phức tạp
    if 'categories' in df.columns:
        def extract_cat(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_cat)
        
        # Bỏ cột categories gốc đi cho nhẹ nếu không cần dùng nữa
        df = df.drop(columns=['categories'])

    # Ép kiểu chuỗi để đảm bảo không bị lỗi schema khi lưu Parquet
    for col in ['parent_asin', 'title', 'store', 'main_category', 'description']:
        if col in df.columns:
            df[col] = df[col].astype(str)

    return df.reset_index(drop=True)

def load_and_clean_meta_chunked(filepath, chunk_size=50_000):
    TEMP_META_DIR = PROCESSED_DIR / "_temp_meta_chunks"
    TEMP_META_DIR.mkdir(exist_ok=True)

    # Xóa chunks cũ nếu có
    for f in TEMP_META_DIR.glob("*.parquet"):
        f.unlink()

    chunk_id      = 0
    total_loaded  = 0
    current_chunk = []
    
    print(f"Bắt đầu xử lý Meta file. Chunk size: {chunk_size:,} dòng/lần\n")

    # Thay gzip.open bằng open
    with open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading meta", unit=" lines"):
            try:
                current_chunk.append(json.loads(line.strip()))
            except:
                continue

            if len(current_chunk) >= chunk_size:
                df = pd.DataFrame(current_chunk)
                df = _clean_meta_df(df)          
                if df is not None:
                    df.to_parquet(TEMP_META_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
                    total_loaded += len(df)
                chunk_id += 1
                current_chunk = []
                del df
                gc.collect()

    # Xử lý phần dư
    if current_chunk:
        df = pd.DataFrame(current_chunk)
        df = _clean_meta_df(df)
        if df is not None:
            df.to_parquet(TEMP_META_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
            total_loaded += len(df)
        del df
        gc.collect()

    # Ghép chunks lại
    chunk_files = sorted(TEMP_META_DIR.glob("*.parquet"))
    writer = None
    output = PROCESSED_DIR / "meta_clean.parquet"

    print(f"\nĐang gộp {len(chunk_files)} chunks meta...")
    
    for cf in tqdm(chunk_files, desc="Merging"):
        table = pq.read_table(cf)
        if writer is None:
            writer = pq.ParquetWriter(output, table.schema, compression='snappy')
        writer.write_table(table)
        del table
        gc.collect()

    if writer:
        writer.close()

    # Xóa temp
    if TEMP_META_DIR.exists():
        shutil.rmtree(TEMP_META_DIR)

    if output.exists():
        sz = output.stat().st_size / (1024**2)
        print(f"\nHoàn tất! Đã lưu: {output}")
        print(f"Tổng số dòng: {total_loaded:,}")
        print(f"Dung lượng meta_clean.parquet: {sz:.2f} MB")
    

    # print(f" {ram_usage()}")

# Chạy hàm
load_and_clean_meta_chunked(META_PATH, chunk_size=50_000)

Bắt đầu xử lý Meta file. Chunk size: 50,000 dòng/lần



Reading meta: 3735584 lines [02:51, 21745.40 lines/s]



Đang gộp 0 chunks meta...


Merging: 0it [00:00, ?it/s]


Hoàn tất! Đã lưu: F:\amazon_data\data\processed\meta_clean.parquet
Tổng số dòng: 0
Dung lượng meta_clean.parquet: 1125.59 MB


In [ ]:
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# 1. Kiểm tra và load bảng Meta
meta_path = PROCESSED_DIR / "meta_clean.parquet"
if not meta_path.exists():
    raise FileNotFoundError(f"Không tìm thấy bảng meta tại: {meta_path}")

# Lọc cột một cách gọn gàng, không cần check đường dẫn dài dòng bên trong list comprehension
cols_to_load = ['parent_asin', 'title', 'price', 'main_category']
meta_for_merge = pd.read_parquet(meta_path, columns=cols_to_load)

gc.collect()
print(f"Loaded meta: {meta_for_merge.shape}")
# print(f"RAM: {ram_usage()}") # Mở comment nếu bác có hàm này

# 2. Setup file Review và Output
review_pf     = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
merged_output = PROCESSED_DIR / "merged_clean.parquet"

writer_m      = None
total_merged  = 0
n_rg          = review_pf.metadata.num_row_groups

print(f"Bắt đầu merge {n_rg} row groups...\n")

# 3. Đọc từng Row Group, Merge và Ghi nối tiếp
for i in tqdm(range(n_rg), desc="Merging"):
    # Đọc chunk từ file review gốc
    chunk = review_pf.read_row_group(i).to_pandas()

    merged = chunk.merge(meta_for_merge, on='parent_asin', how='left')

    # Xử lý trùng tên cột: Pandas sẽ tự đổi thành title_x (của review) và title_y (của meta)
    if 'title_x' in merged.columns and 'title_y' in merged.columns:
        merged = merged.rename(columns={
            'title_x': 'review_title',
            'title_y': 'product_title'
        })
    elif 'title' in merged.columns:
        # Đề phòng trường hợp file review mất cột title, Pandas không tự thêm hậu tố _x _y
        merged = merged.rename(columns={'title': 'product_title'})

    # Chuyển lại thành PyArrow Table
    table = pa.Table.from_pandas(merged, preserve_index=False)
    
    # Khởi tạo writer ở vòng lặp đầu tiên
    if writer_m is None:
        writer_m = pq.ParquetWriter(merged_output, table.schema, compression='snappy')
        
    writer_m.write_table(table)
    total_merged += len(merged)

    # Dọn dẹp RAM triệt để
    del chunk, merged, table
    gc.collect()

# Đóng file an toàn
if writer_m:
    writer_m.close()

# Báo cáo kết quả
if merged_output.exists():
    sz = merged_output.stat().st_size / (1024**3)
    print(f"\nHoàn tất! File cuối cùng đã lưu tại: {merged_output}")
    print(f"merged_clean.parquet: {total_merged:,} rows | {sz:.2f} GB")
else:
    print("\nCó lỗi xảy ra, không tìm thấy file output.")

Loaded meta: (3735584, 4)
Bắt đầu merge 675 row groups...



Merging:   5%|▍         | 33/675 [01:52<37:04,  3.47s/it]

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter
from tqdm import tqdm

# --- CẤU HÌNH ---
# MIN_REVIEWS_PER_USER = 5
# MIN_REVIEWS_PER_ITEM = 5

review_pf = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
n_rg      = review_pf.metadata.num_row_groups

# ==========================================
# PASS 1: Đếm user/item counts
# ==========================================
print("Pass 1/2: Đếm user/item counts...")
user_counter = Counter()
item_counter = Counter()

for i in tqdm(range(n_rg), desc="Counting"):
    chunk = review_pf.read_row_group(i, columns=['user_id','parent_asin']).to_pandas()
    user_counter.update(chunk['user_id'].value_counts().to_dict())
    item_counter.update(chunk['parent_asin'].value_counts().to_dict())
    
    del chunk
    gc.collect()

# Lấy ra tập hợp (set) các user/item thỏa mãn điều kiện
valid_users = {u for u, c in user_counter.items() if c >= MIN_REVIEWS_PER_USER}
valid_items = {p for p, c in item_counter.items() if c >= MIN_REVIEWS_PER_ITEM}

print(f"   Valid users: {len(valid_users):,} / {len(user_counter):,}")
print(f"   Valid items: {len(valid_items):,} / {len(item_counter):,}")


# ==========================================
# PASS 2: Lọc & Lưu
# ==========================================
print("\nPass 2/2: Lọc và lưu...")
rec_output = PROCESSED_DIR / "review_for_rec.parquet"
writer_r   = None
total_rec  = 0

for i in tqdm(range(n_rg), desc="Filtering"):
    chunk = review_pf.read_row_group(i).to_pandas()
    
    # Giữ lại các dòng thỏa mãn CẢ 2 điều kiện
    filtered = chunk[
        chunk['user_id'].isin(valid_users) &
        chunk['parent_asin'].isin(valid_items)
    ]
    
    if len(filtered) > 0:
        table = pa.Table.from_pandas(filtered, preserve_index=False)
        if writer_r is None:
            writer_r = pq.ParquetWriter(rec_output, table.schema, compression='snappy')
        writer_r.write_table(table)
        total_rec += len(filtered)
        del table
        
    del chunk, filtered
    gc.collect()

if writer_r: 
    writer_r.close()

# Kiểm tra kết quả
if rec_output.exists():
    sz = rec_output.stat().st_size / (1024**2) # Đổi ra MB vì file sau khi lọc thường nhỏ đi nhiều
    print(f"\nreview_for_rec.parquet: {total_rec:,} rows | {sz:.1f} MB")

# Nhớ mở comment nếu bạn có hàm ram_usage()
print(f" {ram_usage()}")

## 4 · Build Item Vocab
Map `parent_asin` (string) → `item_idx` (int) — cần thiết cho Item2Vec và GRU4Rec.

In [ ]:
# ── item vocab ────────────────────────────────────────────────────────
# Sắp xếp theo frequency giảm dần → idx nhỏ = item phổ biến hơn
item_freq  = df["parent_asin"].value_counts()   # đã sort descending
item2idx   = {item: idx for idx, item in enumerate(item_freq.index)}
idx2item   = {idx: item for item, idx in item2idx.items()}
N_ITEMS    = len(item2idx)

# ── user vocab ────────────────────────────────────────────────────────
user_freq  = df["user_id"].value_counts()
user2idx   = {u: i for i, u in enumerate(user_freq.index)}
N_USERS    = len(user2idx)

# Gán idx vào df
df["item_idx"] = df["parent_asin"].map(item2idx)
df["user_idx"] = df["user_id"].map(user2idx)

# Gán vào meta
df_meta["item_idx"] = df_meta["parent_asin"].map(item2idx)

print(f"Vocab size  : {N_ITEMS:,} items  |  {N_USERS:,} users")

# Lưu vocab
import json
vocab_dir = ROOT_DIR / "data" / "vocab"
vocab_dir.mkdir(exist_ok=True)
with open(vocab_dir / "item2idx.json", "w") as f:
    json.dump(item2idx, f)
with open(vocab_dir / "idx2item.json", "w") as f:
    json.dump({str(k): v for k, v in idx2item.items()}, f)
print(f"Vocab saved → {vocab_dir}")

## 5 · Build User Sessions
Group by `user_id`, sort theo `timestamp` tăng dần → ra sequence `[item_idx_0, item_idx_1, ...]`.

In [ ]:
log.info("Building sessions...")

# Sort theo (user, time)
df_sorted = df.sort_values(["user_idx", "timestamp"], ascending=True)

# Group → list of item indices per user
sessions = (
    df_sorted.groupby("user_idx", sort=False)["item_idx"]
    .apply(list)
    .reset_index()
    .rename(columns={"item_idx": "item_seq"})
)

# Cắt sequence quá dài (giữ MAX_SEQ_LEN cái mới nhất)
sessions["item_seq"] = sessions["item_seq"].apply(
    lambda seq: seq[-MAX_SEQ_LEN:] if len(seq) > MAX_SEQ_LEN else seq
)
sessions["seq_len"] = sessions["item_seq"].apply(len)

print(f"Total sessions : {len(sessions):,}")
print(f"Sequence length:")
print(sessions["seq_len"].describe().to_string())
print(ram())

## 6 · Temporal Train / Val / Test Split
**Leave-One-Out (LOO) temporal:**  
- `test`  = item **cuối cùng** trong sequence mỗi user  
- `val`   = item **áp cuối**  
- `train` = tất cả phần còn lại  

Đây là chuẩn phổ biến nhất trong session-based recommendation.

In [ ]:
# Chỉ giữ user có seq_len >= 3 (cần ít nhất 1 train + 1 val + 1 test)
sessions = sessions[sessions["seq_len"] >= 3].reset_index(drop=True)
print(f"Sessions with len>=3: {len(sessions):,}")

# ── LOO split ─────────────────────────────────────────────────────────
sessions["train_seq"] = sessions["item_seq"].apply(lambda s: s[:-2])
sessions["val_item"]  = sessions["item_seq"].apply(lambda s: s[-2])
sessions["test_item"] = sessions["item_seq"].apply(lambda s: s[-1])

# Thống kê
total_interactions = sessions["seq_len"].sum()
train_interactions = sessions["train_seq"].apply(len).sum()
print(f"\nSplit summary:")
print(f"  train interactions : {train_interactions:,}  ({train_interactions/total_interactions*100:.1f}%)")
print(f"  val  interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  test interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  total              : {total_interactions:,}")

## 7 · Save outputs

In [ ]:
# ── 7.1 reviews_clean.parquet ─────────────────────────────────────────
clean_cols = ["user_idx", "user_id", "item_idx", "parent_asin",
              "rating", "timestamp", "dt"]
df[clean_cols].to_parquet(CLEANED_DIR / "reviews_clean.parquet",
                           index=False, engine="pyarrow")
print(f"Saved reviews_clean.parquet  ({len(df):,} rows)")

# ── 7.2 meta_clean.parquet ────────────────────────────────────────────
df_meta.to_parquet(CLEANED_DIR / "meta_clean.parquet",
                   index=False, engine="pyarrow")
print(f"Saved meta_clean.parquet     ({len(df_meta):,} rows)")

# ── 7.3 sessions.parquet ─────────────────────────────────────────────
# Lưu dạng: user_idx | train_seq (list) | val_item (int) | test_item (int)
import pickle
sessions_save = sessions[["user_idx", "train_seq", "val_item",
                           "test_item", "seq_len"]].copy()

# Parquet không lưu được list of int natively → dùng pickle cho sessions
with open(CLEANED_DIR / "sessions.pkl", "wb") as f:
    pickle.dump(sessions_save, f)
print(f"Saved sessions.pkl           ({len(sessions_save):,} users)")

# Cũng lưu dạng flat parquet để dễ query
# Flatten: mỗi (user, position) là 1 dòng
records = []
for row in tqdm(sessions_save.itertuples(), total=len(sessions_save), desc="Flatten"):
    for pos, item in enumerate(row.train_seq):
        records.append((row.user_idx, pos, item, "train"))
    records.append((row.user_idx, len(row.train_seq), row.val_item, "val"))
    records.append((row.user_idx, len(row.train_seq)+1, row.test_item, "test"))

df_flat = pd.DataFrame(records, columns=["user_idx", "position", "item_idx", "split"])
df_flat.to_parquet(CLEANED_DIR / "interactions_flat.parquet",
                   index=False, engine="pyarrow")
print(f"Saved interactions_flat.parquet  ({len(df_flat):,} rows)")
del records, df_flat; gc.collect()

# ── 7.4 Stats summary ─────────────────────────────────────────────────
print("\n" + "="*50)
print("  PREPROCESSING COMPLETE")
print("="*50)
print(f"  Users          : {N_USERS:>10,}")
print(f"  Items          : {N_ITEMS:>10,}")
print(f"  Interactions   : {len(df):>10,}")
print(f"  Sessions       : {len(sessions_save):>10,}")
print(f"  Avg seq len    : {sessions_save['seq_len'].mean():>10.2f}")
print(f"  Sparsity       : {1 - len(df)/(N_USERS*N_ITEMS):>10.6f}")
print("="*50)
print(ram())

---
## Notes

| Quyết định | Lý do |
|---|---|
| Dedup giữ review mới nhất | Nếu user review item 2 lần, thông tin mới hơn có giá trị hơn |
| Filter `verified_purchase=True` | Loại spam/bot reviews, giữ tín hiệu chất lượng |
| K-core(5,5) lặp đến hội tụ | Đảm bảo mỗi user/item đủ data để học embedding |
| LOO temporal split | Chuẩn đánh giá phổ biến nhất, không data leakage |
| Cắt sequence > 200 | GRU4Rec tốn VRAM nếu sequence quá dài |
| Lưu cả pkl + flat parquet | pkl cho model training (list), parquet cho EDA/query |

**Output files:**
```
F:\mmd\data\cleaned\
    reviews_clean.parquet
    meta_clean.parquet
    sessions.pkl
    interactions_flat.parquet
F:\mmd\data\vocab\
    item2idx.json
    idx2item.json
```
**→ Notebook 03:** EDA sâu trên cleaned data